# TM1py: Reading Cells

This module is the fourth chunk of the tm1py course. Readers are
assumed to have absorbed the first three chunks: the mental model of
tm1py as a thin REST wrapper, the connect-and-list pattern with
`TM1Service` and `with`, and the structural walk from cube to
element through the metadata services.

The audience remains TM1 expert. MDX is taken for granted as a query
language; the chunk does not teach MDX, it only shows how to hand MDX
to tm1py and what comes back. The same is true of TI's `CellGetN`
and `CellGetS`, which are the natural points of comparison for
single-cell reads.

The goal is narrow on purpose: pull values out of a cube and into
Python, either as a single number or as a DataFrame. Three methods on
`tm1.cells` cover roughly ninety percent of practical use:

In [ ]:
tm1.cells.get_value("Sales Plan", "2026Q1,Europe,Standard,Plan,Revenue")
tm1.cells.execute_view_dataframe("Sales Plan", "2026 Plan by Region")
tm1.cells.execute_mdx_dataframe(mdx_query)

That is the whole skeleton. Each is paired with the situation it suits
best: a single point lookup, a stable saved view, and a dynamic slice
expressed as MDX. A fourth topic introduces `mdxpy`, the Python builder
that produces MDX strings without manual concatenation. tm1py already
depends on it, so it is available the moment tm1py is installed.

For many learners, this is the chunk where tm1py starts paying off.
The value of "extract TM1 data into pandas for downstream analysis" is
delivered here, in three lines of code per slice. Writing back, which
is the next chunk, is the inverse operation; this one stops at the
read.

The topics below are arranged linearly for review. The Sales Plan
model from earlier chunks is the running example.

---

## Topic list

1. The reading scenario
2. The CellService and what it covers
3. Single cells with get_value
4. Saved views as DataFrames
5. Dynamic slices with MDX
6. The DataFrame shape that comes back
7. mdxpy: building MDX without string concatenation
8. Choosing between get_value, view, and MDX
9. Real-world design principles
10. Common mistakes

---

## 1. The reading scenario

The most common single use case for tm1py is "pull a slice of a cube
into pandas, do something with it, optionally write the result back."
The optional write is chunk 5; everything else lives in this chunk.
Reading is also the operation a TM1 administrator already does many
times a day in PAW, Architect, Excel, and Perspectives. The point of
tm1py is to do the same read, programmatically, with the result landing
as Python data instead of a spreadsheet view.

Three flavours of read cover almost every situation. A single cell at
a known address answers point questions: "what is the revenue plan for
Europe in 2026Q1?" A saved view, already configured on the server,
answers stable recurring questions: "what is the 2026 plan by region?"
A dynamic MDX query answers ad hoc questions whose slice depends on
runtime input: "for each of the regions selected by the user, what
were last quarter's actuals?" Each of the three has a corresponding
method on `tm1.cells`, and each returns its result in the form most
useful for the situation.

The running example uses the Sales Plan cube with dimensions Period,
Region, Product, Version, and Measure. A saved view called `2026 Plan
by Region` is assumed to exist on the server; the MDX examples target
the same cube but build the slice in code. None of the reads modifies
anything on the server, so the entire chunk can be tried against a
production cube without risk.

## 2. The CellService and what it covers

Every cell read goes through `tm1.cells`, the CellService introduced
in chunk 2. The service has more than three methods, but three of them
account for the bulk of practical work and are the focus here.

In [ ]:
tm1.cells.get_value(cube_name, element_string)
tm1.cells.execute_view_dataframe(cube_name, view_name)
tm1.cells.execute_mdx_dataframe(mdx)

The remaining methods are variants for situations that the three
above do not cover well. `execute_mdx` and `execute_view` return raw
cellsets as Python dictionaries keyed by element tuple, which is
useful when the result is going to feed a write rather than a
DataFrame. `execute_view_dataframe_pivot` returns a wide DataFrame
matching the view's row/column shape, useful when the result is
destined for a presentation rather than analysis. `execute_mdx_csv`
streams the result to a CSV-shaped string, useful when the next step
is a file. They all live on the same service and follow the same
shape.

The three methods covered here are the ones to learn first. Once the
shape is in muscle memory, the variants are easy to pick up because
they share the same calling convention.

A few characteristics apply to all of them. Each call is one HTTP
round trip. Each returns the cellset as it stood at the moment the
server processed the query, with the snapshot semantics from chunk 1.
Each is read-only, with no side effect on the cube. None of them
caches; calling the same query twice produces two server-side
evaluations.

## 3. Single cells with get_value

The simplest read is one cell at a known address. The method is
`get_value`, and it takes the cube name and a comma-separated element
string in cube dimension order.

In [ ]:
with TM1Service(**creds) as tm1:
    revenue = tm1.cells.get_value(
        cube_name="Sales Plan",
        element_string="2026Q1,Europe,Standard,Plan,Revenue",
    )
    print(revenue)
    # 120000.0

The element string is the canonical TM1 cell address. Each name
identifies an element in the corresponding dimension; the comma is the
separator. The return value is the cell's value: a `float` for numeric
cells, a `str` for string cells, `None` for an empty numeric cell. The
shape mirrors TI's `CellGetN` and `CellGetS` directly and the
intuition transfers.

The dimension order is the cube's. Topic 3 of the previous chunk
covered how to read it from `cube.dimensions`. Constructing element
strings from a Python tuple in code, rather than typing the comma
separated string by hand, avoids ordering bugs:

In [ ]:
with TM1Service(**creds) as tm1:
    cube = tm1.cubes.get("Sales Plan")
    address = ("2026Q1", "Europe", "Standard", "Plan", "Revenue")
    assert len(address) == len(cube.dimensions)

    revenue = tm1.cells.get_value(
        cube_name="Sales Plan",
        element_string=",".join(address),
    )

`get_value` is the right method when the read is a small, fixed number
of point lookups. For an interactive REPL session checking three
specific cells, it is the natural call. For looping over thousands of
cells, it is the wrong shape: each call is one HTTP round trip, and
the round trip cost dominates. Anything that touches more than a
handful of cells should be expressed as a view or an MDX query and
read in one batched call.

The same caution applies that applied to TI's `CellGetN`: it works
correctly on a consolidated element, returning the consolidated value
rather than an error. This is occasionally what is wanted; more often
it hides a bug where the caller intended a leaf and silently got a
parent's roll-up instead.

## 4. Saved views as DataFrames

A saved view on the server is a contract: it names a slice of a cube,
including row and column axes, title elements, and any element
filters. Reading a view returns the slice that the view defines, with
no additional parameters needed in the calling code.

In [ ]:
with TM1Service(**creds) as tm1:
    df = tm1.cells.execute_view_dataframe(
        cube_name="Sales Plan",
        view_name="2026 Plan by Region",
        private=False,
    )

df.head()
#    Period   Region    Product   Version  Measure   Value
# 0  2026Q1   Europe    Standard  Plan     Revenue   120000.0
# 1  2026Q1   Europe    Standard  Plan     Units       1200.0
# 2  2026Q1   Americas  Standard  Plan     Revenue   250000.0
# ...

The arguments are the cube name, the view name, and a `private` flag.
Public views are visible to everyone with permission on the cube;
private views are scoped to the user who created them. Most automation
reads from public views, so `private=False` is the default and the
typical setting.

The benefit of reading from a view rather than from inline MDX is
governance. The view is visible in PAW or Architect; a TM1
administrator can see what the script is reading without opening
Python. A change to the slice (add a region, switch to a different
period) lives in one place, the view definition, and propagates to
every script that uses the view. The script itself does not need to
change, and the next run picks up the new slice automatically.

The trade off is rigidity. A view that is hand-edited each month to
reset the period is a maintenance burden; a view whose slice depends
on a runtime decision is impossible to express. For those situations,
the next topic, MDX, is the right tool. For everything stable and
recurring, named views are the cleanest input.

The DataFrame returned by `execute_view_dataframe` is in tidy form:
one column per dimension, plus a single `Value` column. The shape is
covered in topic 6.

## 5. Dynamic slices with MDX

When the slice is parameterized, ad hoc, or simply not worth promoting
to a saved view, MDX is the right input. The method
`execute_mdx_dataframe` takes an MDX query string and returns the
result as a tidy DataFrame, the same shape as `execute_view_dataframe`.

In [ ]:
mdx = """
SELECT
    NON EMPTY {[Period].[2026Q1], [Period].[2026Q2]} ON ROWS,
    NON EMPTY {[Region].[Europe], [Region].[Asia]}   ON COLUMNS
FROM [Sales Plan]
WHERE ([Measure].[Revenue], [Version].[Plan], [Product].[Standard])
"""

with TM1Service(**creds) as tm1:
    df = tm1.cells.execute_mdx_dataframe(mdx)

df.head()
#    Period   Region   Value
# 0  2026Q1   Europe   120000.0
# 1  2026Q1   Asia      90000.0
# 2  2026Q2   Europe   135000.0
# 3  2026Q2   Asia     110000.0

MDX is the canonical TM1 query language and is identical to the MDX
that TM1 evaluates from PAW, Excel, or any other client. tm1py does
not transform or rewrite the query; the string is passed to the server
verbatim, evaluated there, and the result returned as JSON, which the
library parses into a DataFrame.

Parameterizing an MDX string from Python is straightforward but
deserves caution. f-strings or `str.format` work for the cases where
parameters are known good values: a period, a region, a measure
selected from a small fixed list. For values constructed from user
input or external sources, manually building MDX raises the same
escaping concerns as building any other string-based query language.
The next topic, `mdxpy`, exists in part to remove that concern by
constructing the MDX from typed Python pieces.

A useful keyword on `execute_mdx_dataframe` is `top`. Setting
`top=1000` caps the returned cellset at 1000 rows. For exploratory
queries against a large cube, this is the difference between a
five-second response and a twenty-minute one.

In [ ]:
df = tm1.cells.execute_mdx_dataframe(mdx, top=1000)

The cap is enforced by the server-side cellset; the result is the
first 1000 rows of the query's natural ordering, not a random sample.
For a true random sample, the right approach is an MDX `RANDOM`
expression in the query itself.

## 6. The DataFrame shape that comes back

Both `execute_view_dataframe` and `execute_mdx_dataframe` return their
result in the same tidy shape: one column per dimension that varies in
the cellset, plus a single `Value` column. Dimensions held fixed by a
WHERE clause or by a view's title elements are not in the DataFrame,
because their value is constant across every row.

In [ ]:
df.columns
# Index(['Period', 'Region', 'Value'], dtype='object')

df.dtypes
# Period    object
# Region    object
# Value     float64
# dtype: object

The dimension columns are `object` dtype carrying strings; the `Value`
column is `float64` for numeric measures, or `object` for string
measures, or mixed when the cellset contains both. Empty cells appear
as `NaN` rather than zero, which matters for any subsequent arithmetic
that does not want to treat empty as zero.

The tidy shape is the form most directly useful for pandas. Every
dimension is a column, every measure value is a row, and standard
operations like `groupby`, `pivot`, `merge`, and `query` all work as
expected. Pandera schemas (covered in the pandera module) attach
naturally to this shape: one Column per dimension with an `isin`
check, one Column for `Value` with a range check, and the shape is
documented and validated in one place.

For situations where a wide pivot is wanted instead, the alternative
method is `execute_view_dataframe_pivot`, which returns the result in
the shape suggested by the view's row and column axes (one row per row
axis tuple, one column per column axis tuple). The pivot form is
useful when the immediate next step is presentation; for analytical
work, the tidy form is almost always easier and the wide form can be
produced from it with `df.pivot` when needed.

## 7. mdxpy: building MDX without string concatenation

Writing MDX as Python f-strings works for fixed queries. For queries
parameterized at runtime, especially when the parameters are
collections (a set of regions selected from a UI, a list of periods
generated from a date range), string concatenation becomes fragile.
Trailing commas, mismatched brackets, missing escapes, and accidental
typos all produce MDX that the server rejects with errors that point
at the wrong line.

`mdxpy` is a small Python library that builds MDX from typed pieces:
`Member`, `MdxHierarchySet`, `MdxBuilder`. It comes installed with
tm1py automatically (tm1py depends on it internally) and produces
plain MDX strings that go straight into `execute_mdx_dataframe`.

In [ ]:
from mdxpy import MdxBuilder, MdxHierarchySet, Member

regions = ["Europe", "Asia", "Americas"]
periods = ["2026Q1", "2026Q2"]

mdx = (
    MdxBuilder.from_cube("Sales Plan")
    .rows_non_empty()
    .add_hierarchy_set_to_row_axis(
        MdxHierarchySet
        .members([Member.of("Period", p) for p in periods])
    )
    .columns_non_empty()
    .add_hierarchy_set_to_column_axis(
        MdxHierarchySet
        .members([Member.of("Region", r) for r in regions])
    )
    .where(
        Member.of("Measure",  "Revenue"),
        Member.of("Version",  "Plan"),
        Member.of("Product",  "Standard"),
    )
    .to_mdx()
)

with TM1Service(**creds) as tm1:
    df = tm1.cells.execute_mdx_dataframe(mdx)

Each piece is a Python object with a known type. `Member.of(dimension,
element)` produces one element reference. `MdxHierarchySet.members([...])`
produces a set of them. `MdxBuilder` assembles axes and a WHERE
clause. `to_mdx()` renders the whole thing as a string.

For a TM1 administrator already fluent in MDX, plain MDX strings
remain a fine choice for fixed queries. The case for `mdxpy` is
exactly where strings are weakest: parameter sweeps, programmatic
construction of large axes, and queries that need to escape element
names containing commas or brackets. The price is one extra import
and a slightly more verbose call site; the benefit is that the
resulting string is correct by construction.

A useful side property: an MDX query produced by `mdxpy` can be
printed and inspected before sending. Debugging a parameterized MDX
query becomes "what string did the builder produce" rather than
"what does my f-string look like after substitution," and the answer
is one `print(mdx)` away.

## 8. Choosing between get_value, view, and MDX

The three methods are not interchangeable; each has a situation it
suits best.

`get_value` is the right call for a small, known number of point
lookups. The shape mirrors TI's `CellGetN`. It is wrong for any loop
over more than a handful of cells, where the per-call HTTP cost
dominates. A loop calling `get_value` ten thousand times is one of the
clearest performance bugs in tm1py and one of the easiest to write
without realizing.

`execute_view_dataframe` is the right call when the slice is stable
and recurring. The view, defined on the server, is the contract; the
script names it. Maintenance lives in one place, visible to TM1
administrators in PAW. Every recurring report, every monthly extract,
every dashboard data feed is well served by this shape.

`execute_mdx_dataframe` is the right call when the slice is dynamic
or parameterized at runtime. The query is built in Python, optionally
with `mdxpy`, and submitted as a string. Maintenance lives in the
script; visibility from outside Python is correspondingly lower, but
flexibility is higher.

The three are also not the only options. For a cellset that is going
to be transformed cell-by-cell rather than as a DataFrame,
`execute_view` and `execute_mdx` return a dictionary keyed by element
tuple. For pivot-shaped output,
`execute_view_dataframe_pivot` matches the view's row/column layout.
For a streaming CSV destination, `execute_mdx_csv`. The decision tree
between the three primary methods covers the common cases; the
variants are reached for when their specific shape is a better fit.

## 9. Real-world design principles

**Read in batches, never in loops.** A `get_value` call inside a
Python loop pays the HTTP round trip per iteration. The same data,
fetched as a single MDX or view query, is one round trip total and is
between one and three orders of magnitude faster for any non-trivial
slice. The exception is the genuinely small case where the loop runs
five times against five known addresses; for anything larger,
batching is mandatory.

**Prefer named views for stable, recurring slices.** A view is a
contract that lives in one place, visible in PAW, editable without
touching code. The first time the same MDX appears in two different
scripts, promote it to a view. The second time, it should already
have been a view from the start.

**Reach for MDX when the slice is dynamic.** A view cannot encode
"the regions the user picked at runtime" or "every period between A
and B where A and B come from arguments." Those are MDX situations.
For programmatically constructed MDX, `mdxpy` removes the string
escaping concerns and produces a query that is correct by
construction.

**Cap exploratory queries with `top`.** A query against a large cube
without `NON EMPTY` and without `top` can return millions of rows and
exhaust memory or block on the server for a long time. During
development, set `top=1000` and remove the cap only after the slice
is known to be small.

**Take the dimension order from `cube.dimensions`.** When constructing
element strings for `get_value`, build the tuple from
`cube.dimensions` rather than hardcoding the order in source. This is
the rule from chunk 3, restated for the cell read context: the cube
is the authoritative source of dimension order, not the script.

**Treat empty cells as NaN, not zero.** Empty TM1 cells become `NaN`
in the returned DataFrame. Code that downstream depends on summing or
averaging needs to decide explicitly whether `NaN` should be filled
with zero (`df["Value"].fillna(0)`) or left as missing. The default
pandas behaviour, propagating `NaN` through arithmetic, is usually
correct; assuming it does not need thought is the bug.

## 10. Common mistakes

A short collection of errors that come up while learning to read cells
through tm1py.

**Looping `get_value` over many cells.** The canonical performance
bug. Each iteration is one HTTP round trip; ten thousand iterations
are ten thousand round trips. A view or MDX query covers the same
ground in one trip.

In [ ]:
# Wrong: one round trip per cell
totals = []
for region in regions:
    for product in products:
        v = tm1.cells.get_value(
            "Sales Plan",
            f"2026Q1,{region},{product},Plan,Revenue",
        )
        totals.append((region, product, v))

# Correct: one MDX, one round trip
mdx = """
SELECT NON EMPTY {[Region].Members} * {[Product].Members} ON ROWS,
       NON EMPTY {[Measure].[Revenue]} ON COLUMNS
FROM [Sales Plan]
WHERE ([Period].[2026Q1], [Version].[Plan])
"""
df = tm1.cells.execute_mdx_dataframe(mdx)

**Hardcoding dimension order in the element string.** The order is
on the cube. Hardcoding it duplicates the metadata and breaks silently
when the cube changes.

In [ ]:
# Wrong: order assumed in code
addr = "2026Q1,Europe,Standard,Plan,Revenue"
v = tm1.cells.get_value("Sales Plan", addr)

# Correct: build the address from cube.dimensions
cube = tm1.cubes.get("Sales Plan")
elements = {
    "Period":  "2026Q1",
    "Region":  "Europe",
    "Product": "Standard",
    "Version": "Plan",
    "Measure": "Revenue",
}
addr = ",".join(elements[d] for d in cube.dimensions)
v = tm1.cells.get_value("Sales Plan", addr)

**Building MDX with f-strings for collections.** A list of regions
inserted with `", ".join(...)` works until the first region name
contains a comma, an apostrophe, or any other MDX special character.
`mdxpy` handles those cases correctly.

In [ ]:
# Wrong: brittle for arbitrary element names
regions = ["Europe", "Asia", "North, Central"]   # contains a comma
mdx = f"""
SELECT NON EMPTY {{[Region].[{'], [Region].['.join(regions)}]}} ON ROWS,
...
"""
# the embedded comma in 'North, Central' breaks the MDX

# Correct: mdxpy quotes and escapes safely
from mdxpy import MdxBuilder, MdxHierarchySet, Member
mdx = (
    MdxBuilder.from_cube("Sales Plan")
    .add_hierarchy_set_to_row_axis(
        MdxHierarchySet.members([Member.of("Region", r) for r in regions])
    )
    .to_mdx()
)

**Forgetting `NON EMPTY` on large queries.** A query over a sparse
cube without `NON EMPTY` returns one row per element tuple, including
empty cells. The result can be millions of rows of `NaN`. `NON EMPTY`
on each axis makes the server skip empty cells before returning the
cellset.

In [ ]:
# Wrong: returns every tuple, mostly empty
mdx = "SELECT {[Region].Members} ON ROWS, {[Period].Members} ON COLUMNS FROM [Sales Plan]"

# Correct: NON EMPTY on both axes
mdx = """
SELECT NON EMPTY {[Region].Members}  ON ROWS,
       NON EMPTY {[Period].Members}  ON COLUMNS
FROM [Sales Plan]
"""

**Treating `NaN` as zero by accident.** Pandas operations propagate
`NaN`. A sum of a column containing `NaN` is `NaN`, not the sum of the
non-missing values, unless the operation is told otherwise.

In [ ]:
# Wrong: total is NaN if any cell is empty
total = df["Value"].sum()        # NaN

# Either: skipna is True by default for sum, so this works
total = df["Value"].sum()        # ignores NaN, returns sum of present values
# but for downstream arithmetic, decide explicitly:
df["Value"] = df["Value"].fillna(0)

**Reading from a private view from another user's session.**
`private=True` scopes the read to the calling user's private views.
Asking for a colleague's private view returns "not found." For
shared automation, views must be public.

In [ ]:
# Wrong: tries to read a colleague's private view
df = tm1.cells.execute_view_dataframe(
    "Sales Plan", "Annas Review", private=True,
)
# raises: View not found

# Correct: use a public view
df = tm1.cells.execute_view_dataframe(
    "Sales Plan", "2026 Plan by Region", private=False,
)

**Caching nothing, or caching too aggressively.** A read is a network
round trip; calling the same read twice in succession sends two
queries. A read is also a snapshot; caching the result for a long
time means working from stale data when the underlying cube has
changed. The right cache window is "the duration of one logical unit
of work," typically one script run or one notebook session, and the
right cache is a Python variable.